In [ ]:
import pandas as pd
from neo4j import GraphDatabase, RoutingControl
import time 
import networkx as nx
import matplotlib.pyplot as plt
import random

## NEO4J

In [ ]:
URI = "neo4j://localhost:7687"
AUTH=('neo4j', "M@userMaria")


In [ ]:
def insert_simulated_likes_neo4j(uri, auth, n, database="neo4j"):
    df = pd.read_csv("cpa.csv")
    sac_codes = df["SAC"].dropna().astype(int).unique()


    simulated_data = [
        (random.randint(1, 10000), int(random.choice(sac_codes)))
        for _ in range(n)
    ]

    cypher_query = """
    UNWIND $pairs AS pair
    MERGE (u:User {id: pair.user_id})
    WITH u, pair
    MATCH (s:SAC {sac: pair.sac_code})
    MERGE (u)-[:LIKES]->(s)
    """


    with GraphDatabase.driver(uri, auth=auth) as driver:
        driver.execute_query(
            cypher_query,
            parameters_={
                "pairs": [{"user_id": u, "sac_code": s} for u, s in simulated_data]
            },
            database_=database,
            routing_=RoutingControl.WRITE
        )
        print("Inserted 1000 simulated (User)-[:LIKES]->(SAC) relationships.")



def insert_simulated_users_neo4j(uri, auth, n, database="neo4j"):
    simulated_data = [
        (random.randint(1, 10000), random.randint(1, 10000))
        for _ in range(n)
    ]


    cypher_query = """
    UNWIND $pairs AS pair
    MERGE (u:User {id: pair.user_id1})
    WITH u, pair
    MERGE (s:User {id: pair.user_id2})
    MERGE (u)-[:SIMILAR_TO]->(s)
    """


    with GraphDatabase.driver(uri, auth=auth) as driver:
        driver.execute_query(
            cypher_query,
            parameters_={
                "pairs": [{"user_id1": u, "user_id2": s} for u, s in simulated_data]
            },
            database_=database,
            routing_=RoutingControl.WRITE
        )
        print("Inserted 1000 simulated (User)-[:SIMILAR_TO]->(SAC) relationships.")

insert_simulated_likes_neo4j(URI, AUTH, 50000)
insert_simulated_users_neo4j(URI, AUTH, 50000)

In [ ]:
def get_recommendations(type_name, keywords):
    met_values = ['4','5','6']
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        records, _, _ = driver.execute_query(
            '''
            MATCH (t:Type {name: $type_name})<-[:HAS_TYPE]-(s:SAC)
                  -[:HAS_MET]->(m:MET),
                  (s)-[:HAS_KEYWORD]->(k:Keyword)
            WHERE k.name IN $keywords
              AND m.name IN $met_values
              AND EXISTS {
                MATCH (:User {id: 1})-[:SIMILAR_TO]->(:User)-[:LIKES]->(s)
              }
            RETURN t, s, m, k
            ''',
            parameters_={
                "type_name": type_name,
                "keywords": keywords,
                "met_values": met_values
            },
            database_="neo4j",
            routing_=RoutingControl.READ
        )
        print(records)
        return records


In [ ]:
start = time.time()
records = get_recommendations('Bicycling', ['Indoor', 'Urban'])
end = time.time()

print(f"Elapsed time: {end - start} seconds")

## MySQL

In [ ]:
import mysql.connector
import random

def insert_simulated_likes_mysql(table_name, n):
    df = pd.read_csv("cpa.csv")
    sac_codes = df["SAC"].dropna().unique()

    simulated_data = [
        (random.randint(1, 100000), int(random.choice(sac_codes)))
        for _ in range(n)
    ]

    conn = mysql.connector.connect(
        host="localhost",
        port=3036,
        user="root",
        password="", 
        database="Biobank"
    )
    cursor = conn.cursor()

    insert_query = f"""
    INSERT INTO {table_name} (user, SAC)
    VALUES (%s, %s)
    """
    cursor.executemany(insert_query, simulated_data)

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {n} simulated rows into '{table_name}'.")


insert_simulated_likes_mysql("likes", 50000)

In [ ]:
import mysql.connector
import random

def insert_simulated_users_mysql(table_name, n):
    
    simulated_data = [
        (random.randint(1, 100000), random.randint(1, 100000))
        for _ in range(n)
    ]

    conn = mysql.connector.connect(
        host="localhost",
        port=3036,
        user="root",
        password="", 
        database="Biobank"
    )
    cursor = conn.cursor()

    insert_query = f"""
    INSERT INTO {table_name} (User1, User2)
    VALUES (%s, %s)
    """
    cursor.executemany(insert_query, simulated_data)

    conn.commit()
    cursor.close()
    conn.close()
    print(f"Inserted {n} simulated rows into '{table_name}'.")


insert_simulated_users_mysql("similar_users", 50000)

In [ ]:
import mysql.connector
import pandas as pd

def get_recommendations_sql(type_name, keywords):
    met_values = ['4', '5', '6']
    placeholders_kw = ', '.join(['%s'] * len(keywords))
    placeholders_met = ', '.join(['%s'] * len(met_values))
    
    query = f"""
    SELECT *
    FROM cpa
    WHERE Type = %s
      AND Keyword IN ({placeholders_kw})
      AND MET IN ({placeholders_met})
      AND SAC IN (
          SELECT SAC
          FROM likes
          WHERE USER IN (
              SELECT User2
              FROM similar_users
              WHERE User1 = 1
          )
      )
    """
    
    params = [type_name] + keywords + met_values
    
    conn = mysql.connector.connect(
        host="localhost",
        port=3036,
        user="root",
        password="", 
        database="Biobank"
    )
    df = pd.read_sql(query, conn, params=params)
    print(df)
    conn.close()
    return df


In [ ]:
start = time.time()
get_recommendations_sql('Bicycling', ['Indoor', 'Urban'])
end = time.time()

print(f"Elapsed time: {end - start} seconds")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Data
mariadb = [(1,0.006), (1000, 0.009), (10000, 0.033), (20000, 0.068), (50000, 0.156), (100000, 0.24)]
neo4j = [(1,0.010), (1000, 0.014), (10000, 0.016), (20000, 0.019), (50000, 0.024), (100000, 0.030)]

# Prepare DataFrame
data = {
    "Number of similar users/likes": [x for x, y in mariadb] + [x for x, y in neo4j],
    "time (in seconds)": [y for x, y in mariadb] + [y for x, y in neo4j],
    "Database": ["MariaDB"] * len(mariadb) + ["Neo4j"] * len(neo4j)
}

df = pd.DataFrame(data)

# Plot
sns.set(style="whitegrid")
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x="Number of similar users/likes", y="time (in seconds)", hue="Database", marker="o")

plt.title("Performance Comparison")
plt.xscale('log')
plt.show()
